## Software Design Patterns in Python

### **Day 4: Structural and Behavioral Patterns**

#### **Module 5: Interface & Composition Patterns**

* **Adapter:** Wrapping legacy code/libraries (Duck typing vs. ABCs).  
* **Facade:** Simplifying complex internal subsystems for external consumers.  
* **Bridge:** Decoupling abstraction from implementation (Composition over Inheritance).

#### **Module 6: Performance & Proxy Patterns**

* **Proxy:** Lazy initialization, Remote Proxies, and Protection Proxies using `__getattr__`.  
* **Flyweight:** Using `__get__`, `__set__` and Factory pools to manage massive object counts.  
* **Composite:** Handling recursive tree structures (filesystems, GUI hierarchies).

#### **Module 7: Event & State Patterns**

* **Observer:** Implementing Pub/Sub systems using Callbacks and Event objects.  
* **State:** Replacing massive if/elif blocks with polymorphic State classes.  
* **Memento:** Capturing and restoring object internal state (Undo/Redo mechanisms).



---

#### Examples of Builder pattern

In [6]:
from urllib.parse import urlparse, urlunparse, urlencode

urlparse_result = urlparse("https://www.example.com/path/to/resource?query=param#fragment")
print("Scheme:", urlparse_result)
print("Netloc:", urlparse_result.netloc)
print("Path:", urlparse_result.path)
print("Query:", urlparse_result.query)
print("Fragment:", urlparse_result.fragment)


scheme = "https"
netloc = "www.example.com"
path = "/path/to/resource"
params = ""
query = {"query": "param"}
fragment = "section1"

url = urlunparse((scheme, netloc, path, '', urlencode(query), fragment))
print(url)

Scheme: ParseResult(scheme='https', netloc='www.example.com', path='/path/to/resource', params='', query='query=param', fragment='fragment')
Netloc: www.example.com
Path: /path/to/resource
Query: query=param
Fragment: fragment
https://www.example.com/path/to/resource?query=param#section1


In [7]:
query_params = {"key1": "value1", "key2": "value2"}
urlencode(query_params)

'key1=value1&key2=value2'

In [8]:
class URLBuilder:
    def __init__(self):
        self._scheme = "https"
        self._netloc = "localhost"
        self._path = "/"
        self._params = ""
        self._query = {}
        self._fragment = ""

    def scheme(self, s):
        self._scheme = s
        return self
    
    def netloc(self, n):
        self._netloc = n
        return self
    
    def path(self, p):
        self._path = p
        return self
    
    def params(self, p):
        self._params = p
        return self
    
    def query(self, key, value):
        self._query[key] = value
        return self 
    
    def fragment(self, f):
        self._fragment = f
        return self 
    
    def build(self):
        from urllib.parse import urlunparse, urlencode
        query_string = urlencode(self._query)
        return urlunparse((self._scheme, 
                           self._netloc, 
                           self._path, 
                           self._params, 
                           query_string, 
                           self._fragment))

    def __str__(self):
        return self.build()


url = (URLBuilder()
       .scheme("https")
       .netloc("api.example.com")
       .path("/v1/search")
       .query("q", "python")
       .query("page", 2)
       .query("limit", 10)
       .fragment("section1"))

print(url)


https://api.example.com/v1/search?q=python&page=2&limit=10#section1


In [9]:
class URLBuilder:
    def __init__(self):
        self._scheme = "https"
        self._netloc = "localhost"
        self._path = "/"
        self._params = ""
        self._query = {}
        self._fragment = ""

    def __getattr__(self, name):
        if name in ["scheme", "netloc", "path", "params", "fragment"]:
            attr = f"_{name}"
            def setter(value):
                setattr(self, attr, value)
                return self          
            return setter
        elif name == "query":
            def query_setter(key, value):
                self._query[key] = value
                return self
            return query_setter
        else:
            raise AttributeError(f"'URLBuilder' object has no attribute '{name}'")
        
    def build(self):
        from urllib.parse import urlunparse, urlencode
        query_string = urlencode(self._query)
        return urlunparse((self._scheme, 
                           self._netloc, 
                           self._path, 
                           self._params, 
                           query_string, 
                           self._fragment))

    def __str__(self):
        return self.build()
    
url = (URLBuilder()
       .scheme("https")
       .netloc("api.example.com")
       .path("/v1/search")
       .query("q", "python")
       .query("page", 2)
       .query("limit", 10)
       .fragment("section1"))
print(url)


https://api.example.com/v1/search?q=python&page=2&limit=10#section1


In [21]:
class URLBuilder:
    def __init__(self):
        self.scheme = "https"
        self.netloc = "localhost"
        self.path = "/"
        self.params = ""
        self.query = {}
        self.fragment = ""

    def __setitem__(self, key, value):
        if key in ["scheme", "netloc", "path", "params", "fragment"]:
            setattr(self, key, value)
        elif key == "query":
            self.query[key] = value
        else:
            raise KeyError(f"Invalid key: {key}")
    
    def build(self):
        from urllib.parse import urlunparse, urlencode
        query_string = urlencode(self.query)
        return urlunparse((self.scheme, 
                           self.netloc, 
                           self.path, 
                           self.params, 
                           query_string, 
                           self.fragment))

    def __str__(self):
        return self.build()

url = URLBuilder()
url["scheme"] = "https"
url["netloc"] = "api.example.com"
url["path"] = "/v1/search"
url["query"] = {"q": "python", "page": 2, "limit": 10}
url["fragment"] = "section1"
print(url)

https://api.example.com/v1/search?query=%7B%27q%27%3A+%27python%27%2C+%27page%27%3A+2%2C+%27limit%27%3A+10%7D#section1


In [11]:
# Anti-pattern: Building a URL manually without a builder class
# A cleaner pythonic approach towards implementing the Builder pattern

from urllib.parse import urlunparse, urlencode

# Step 1:Build a URL piece by piece
scheme    = "https"
netloc    = "api.example.com"
path      = "/v1/search"
params    = ""
query     = urlencode({"q": "python", "page": 2, "limit": 10})  
fragment  = "section1"

# Step 2: Assemble the URL
url = urlunparse((scheme, netloc, path, params, query, fragment))
print(url)


https://api.example.com/v1/search?q=python&page=2&limit=10#section1


In [ ]:
# Anti-pattern: Building a configuration file manually without a builder class

import configparser

config = configparser.ConfigParser()      # blank builder

config['DEFAULT'] = {                     # add sections one by one
    'debug': 'false',
    'log_level': 'WARNING'
}
config['database'] = {
    'host': 'localhost',
    'port': '5432',
    'name': 'mydb'
}
config['cache'] = {
    'backend': 'redis',
    'timeout': '300'
}

# "build" — write the final product to a file
with open('config.ini', 'w') as f:
    config.write(f)

In [18]:
class Test:
    def __getattr__(self, attr):
        print(f"Accessing attribute: {attr}")
        return lambda x: f"Value of {attr}"
    
t1 = Test()
t2 = Test()
t1.__add__(t2)
t1 + t2

Accessing attribute: __add__


TypeError: unsupported operand type(s) for +: 'Test' and 'Test'

In [19]:
# Anti-pattern: Building a logger manually without a builder class

import logging

# Build the formatter
formatter = logging.Formatter(
    fmt='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Build the handler
handler = logging.StreamHandler()
handler.setLevel(logging.DEBUG)
handler.setFormatter(formatter)       # attach formatter to handler

# Build the logger
logger = logging.getLogger("myapp")
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)            # attach handler to logger

# Now USE the fully constructed logger
logger.info("Application started")
logger.warning("Low memory")

2026-05-14 09:33:48 [INFO] myapp: Application started
2026-05-14 09:33:48 [WARNING] myapp: Low memory


In [ ]:
# Building a logger using a Builder class (also implements a Facade)
import logging
class LoggingBuilder:
    def __init__(self, name):
        self._logger = logging.getLogger(name)
        self._format = {}
        self._handler = None

    def level(self, lvl):
        self._logger.setLevel(lvl)
        return self
    
    def format(self, key, value):
        self._format[key] = value
        return self

    def handler(self, hndlr):
        self._handler = hndlr
        return self

    def build(self):
        if self._format and self._handler:
            self._handler.setFormatter(logging.Formatter(**self._format))
            self._logger.addHandler(self._handler)
        return self._logger

import logging

# Build the logger using the builder
logger_builder = LoggingBuilder("MyLogger")
logger = (logger_builder
          .format('fmt', '%(asctime)s [%(levelname)s] %(name)s: %(message)s')
          .format('datefmt', '%Y-%m-%d %H:%M:%S')
          .level(logging.DEBUG)
          .handler(logging.StreamHandler())
          .build())

# Now USE the fully constructed logger
logger.info("Application started")
logger.warning("Low memory")

2026-05-13 22:13:04 [INFO] MyLogger: Application started
2026-05-13 22:13:04 [WARNING] MyLogger: Low memory


In [ ]:
# Anti-pattern: Building a CLI argument parser manually without a builder class
# Another pythonic approach towards implementing the Builder pattern for a CLI argument parser

import argparse

# Director controls the build sequence
parser = argparse.ArgumentParser(description="My CLI tool")  # foundation

# Add parts step by step
parser.add_argument("filename",   help="Input file",  type=str)
parser.add_argument("--verbose",  help="Verbose mode", action="store_true")
parser.add_argument("--output",   help="Output path",  type=str, default="out.txt")
parser.add_argument("--retries",  help="Max retries",  type=int, default=3)

# "build" — parse_args() produces the final constructed Namespace object
args = parser.parse_args()
print(args)

In [20]:
# Anti-pattern: Building an email message manually without a builder class
# A cleaner pythonic approach towards implementing the Builder pattern for an email message

from email.message import EmailMessage

msg = EmailMessage()                          # start with empty object
msg['Subject'] = 'Hello!'                     # set parts incrementally
msg['From'] = 'sender@example.com'
msg['To'] = 'receiver@example.com'
msg.set_content('This is the email body.')    # add body
msg.add_attachment(b'data', maintype='application', subtype='octet-stream')

print(msg)

Subject: Hello!
From: sender@example.com
To: receiver@example.com
MIME-Version: 1.0
Content-Type: multipart/mixed; boundary="===============8948085110687101095=="

--===============8948085110687101095==
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 7bit

This is the email body.

--===============8948085110687101095==
Content-Type: application/octet-stream
Content-Transfer-Encoding: base64
MIME-Version: 1.0
Content-Disposition: attachment

ZGF0YQ==

--===============8948085110687101095==--



In [ ]:
# Homework Exercise: Implement a LogAnalyzer class using the builder pattern

# Log file: https://files.chandrashekar.info/sshd_minimal.log

class LogAnalyzer:
    def __init__(self, filepath):
        self.filepath = filepath
        # TODO: Initialize other attributes for filtering and extraction
    

log = (LogAnalyzer("sshd_minimal.log")
       .date_from("Dec 10")
       .date_to("Dec 12")
       .ident("sshd")
       .pattern("Invalid user (?P<username>\w+) from (?P<ip_address>\d+\.\d+\.\d+\.\d+)")
       .extract("ip_address")
       .apply())

for record in log:
    print(record)



---

#### Prototype Pattern

#### Prototype Pattern - used to create duplicate of an existing object.

In python, simply using the `copy` module should achieve this.

In [23]:
class Car:
    def __init__(self, make, model, year):
        self.make = make
        self.model = model
        self.year = year
    def __repr__(self):
        return f"{self.year} {self.make} {self.model}"
    
c1 = Car("Toyota", "Camry", 2020)
print(f"{c1=}")

c2 = c1 # Assignment copy references, not objects.
c2.model = "Corolla"
print(f"{c1=}", f"{c2=}", sep="\n")
print(c1 is c2)


c1=2020 Toyota Camry
c1=2020 Toyota Corolla
c2=2020 Toyota Corolla
True


In [ ]:
from copy import deepcopy
c3 = deepcopy(c1)  # Creates a deep copy of the object
c3.model = "RAV4"
print(f"{c1=}", f"{c3=}", sep="\n")
print(f"{c1 is c3=}")

c1=2020 Toyota Corolla
c3=2020 Toyota RAV4
c1 is c3=False


In [25]:
import copy

a = [10, 20, [30, 40]]
b = copy.copy(a)  # Shallow copy
print(a, b, sep="\n")
b[0] = 100
b[2][0] = 300
print("-----")
print(a, b, sep="\n")

[10, 20, [30, 40]]
[10, 20, [30, 40]]
-----
[10, 20, [300, 40]]
[100, 20, [300, 40]]


In [26]:
import copy

a = [10, 20, [30, 40]]
b = copy.deepcopy(a)  # Deep copy
print(a, b, sep="\n")
b[0] = 100
b[2][0] = 300
print("-----")
print(a, b, sep="\n")

[10, 20, [30, 40]]
[10, 20, [30, 40]]
-----
[10, 20, [30, 40]]
[100, 20, [300, 40]]


In [27]:
class Car:
    def __init__(self, make, model, year):
        self.make = make
        self.model = model
        self.year = year

    def __copy__(self):
        print("Creating a shallow copy of Car")
        return Car(self.make, self.model, 0)
    
    def __deepcopy__(self, memo):
        print("Creating a deep copy of Car: memo =", memo)
        memo["make"] = self.make
        memo["model"] = self.model
        memo["year"] = 0
        return Car(memo["make"], memo["model"], memo["year"])

    def __repr__(self):
        return f"{self.year} {self.make} {self.model}"
    
c1 = Car("Toyota", "Camry", 2020)
c2 = copy.deepcopy(c1)
print(f"{c1=}", f"{c2=}", sep="\n")
print(f"{c1 is c2=}")

Creating a deep copy of Car: memo = {}
c1=2020 Toyota Camry
c2=0 Toyota Camry
c1 is c2=False


---

### Structural Patterns

#### Adapter Pattern

This pattern is used mainly as a mechanism to glue non-compatible objects using a standardizing interface.


In [29]:
ins = open("sample.txt")
ins.read(20)

'this is line 1\nthis '

In [30]:
from urllib.request import urlopen
response = urlopen("https://www.python.org/")
data = response.read(20)
data

b'\x1f\x8b\x08\x00\x00\x00\x00\x00\x00\x03\xd5}\xe9r\xe3\xc6\xb2\xe6\x7fG'

In [34]:
class FileParser:
    def __init__(self, ins):
        self.ins = ins
    
    def parse(self):
        return self.ins.read(20)
    
class URLAdapter:
    def __init__(self, url):
        self.url = url
    
    def read(self, length):
        from urllib.request import urlopen
        with urlopen(self.url) as response:
            return response.read(length)
        

file_parser = FileParser(open("sample.txt"))
print(file_parser.parse())

file_parser = FileParser(URLAdapter("https://www.python.org/"))
print(file_parser.parse())

this is line 1
this 
b'\x1f\x8b\x08\x00\x00\x00\x00\x00\x00\x03\xd5}\xe9r\xe3\xc6\xb2\xe6\x7fG'


#### Adapter Pattern examples in Python standard library

In [36]:
a = "This is a test string with a couple of words."

from io import StringIO

fp = FileParser(StringIO(a))
fp.parse()


'This is a test strin'

In [37]:
import io

# TextIOWrapper is an Adapter —
# it wraps a raw binary stream and adapts it to a text interface

binary_stream = io.BytesIO(b"hello\nworld\n")   # Adaptee: speaks bytes
print(f"{binary_stream=}")

text_stream = io.TextIOWrapper(binary_stream, encoding="utf-8")  # Adapter

# Now we can use a text interface on top of a binary source
for line in text_stream:
    print(repr(line))

# 'hello\n'
# 'world\n'

binary_stream=<_io.BytesIO object at 0x10570a160>
'hello\n'
'world\n'


In [41]:
ins = list(open("sample.txt"))

len(ins)

5

In [ ]:
import logging
import logging.handlers

# SocketHandler — adapts a raw TCP socket to the logging.Handler interface
socket_handler = logging.handlers.SocketHandler('localhost', 9999)

# HTTPHandler — adapts an HTTP endpoint to the logging.Handler interface
http_handler = logging.handlers.HTTPHandler(
    host='logs.example.com',
    url='/ingest',
    method='POST'
)

# QueueHandler — adapts a queue.Queue to the logging.Handler interface
import queue
log_queue = queue.Queue()
queue_handler = logging.handlers.QueueHandler(log_queue)

# All three look identical to the logger — pure Adapter
logger = logging.getLogger("myapp")
logger.addHandler(socket_handler)
logger.warning("This goes over TCP")   # logger has no idea it's a socket

In [42]:
import csv, io

raw_csv = "name,age,city\nAlice,30,NYC\nBob,25,LA\n"

reader = csv.reader(io.StringIO(raw_csv))       # returns lists
dict_reader = csv.DictReader(io.StringIO(raw_csv))  # Adapter: same source, dict interface

print("--- csv.reader (raw) ---")
for row in reader:
    print(row)                  # ['name', 'age', 'city'], ['Alice', '30', 'NYC'] ...

print("--- csv.DictReader (adapted) ---")
for row in dict_reader:
    print(row)                  # {'name': 'Alice', 'age': '30', 'city': 'NYC'} ...

--- csv.reader (raw) ---
['name', 'age', 'city']
['Alice', '30', 'NYC']
['Bob', '25', 'LA']
--- csv.DictReader (adapted) ---
{'name': 'Alice', 'age': '30', 'city': 'NYC'}
{'name': 'Bob', 'age': '25', 'city': 'LA'}


In [43]:
from pathlib import Path
import os

path_obj = Path("/usr/local/bin")   # Adaptee: pathlib.Path object
print(path_obj, type(path_obj))     # /usr/local/bin <class 'pathlib.PosixPath'>


/usr/local/bin <class 'pathlib.PosixPath'>


In [46]:
#filename = "sample.txt"
filename = Path("sample.txt")  # Path object can be used directly with open() due to internal adaptation
with open(filename) as f:
    print(f.read())

this is line 1
this is line 2
this is line 3
this is line 4
this is the last line


In [ ]:

# Many older stdlib functions expect str, not Path
# os.fspath() is the Adapter — converts Path → str
path_str = os.fspath(path_obj)
print(path_str, type(path_str))     # /usr/local/bin <class 'str'>

# This is also why open(), os.listdir() etc. accept Path objects —
# they internally call os.fspath() to adapt Path → str
with open(Path("/etc/hostname")) as f: # with open(os.fspath(Path("/etc/hostname"))) --- IGNORE ---
    print(f.read())

In [ ]:
import threading
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def task(n):
    return sum(range(n))

with ThreadPoolExecutor(max_workers=4) as executor:    # threads backend
    futures = [executor.submit(task, i) for i in range(10, 15)]
    print([f.result() for f in futures])

with ProcessPoolExecutor(max_workers=4) as executor:   # processes backend
    futures = [executor.submit(task, i) for i in range(10, 15)]
    print([f.result() for f in futures])


#### Anti-pattern: Using ducktyping as alternative to Adapter pattern 

In [49]:
import threading
import multiprocessing
#from concurrent.futures import ThreadPoolExecutor as Executor
from concurrent.futures import ProcessPoolExecutor as Executor

def task(n):
    from threading import current_thread
    from multiprocessing import current_process
    print(f"Running task({n}) in thread {current_thread().name} of process {current_process().name}")
    return sum(range(n))

with Executor(max_workers=4) as executor:    # threads backend
    futures = [executor.submit(task, i) for i in range(10, 15)]
    print([f.result() for f in futures])


Process SpawnProcess-4:
Process SpawnProcess-1:
Process SpawnProcess-2:
Process SpawnProcess-3:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/lib/python3.12/concurrent/futures/process.py", line 252, in _process_worker
    call_item = call_queue.get(block=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/concurrent/futures/process.py", line 252, in _process_worker
    call

BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

##### Detour - Ducktyping vs Generalization

In [55]:
def square(x):
    return x * x

def apply_map(fn, data, default=None): # Higher-order function that takes a function and data, applies the function to each item in the data
    result = []
    for item in data:
        if type(item) in (int, float):
            ret = fn(item)
        else:
            ret = default
        result.append(ret)
    return result


nums = [3, 7+2j, 5, "hello", 8, 4]
r = apply_map(square, nums, default=0)
print(r)

[9, 0, 25, 0, 64, 16]


In [57]:
# Using generalization approach

from numbers import Number

def square(x):
    return x * x

def to_upper(s):
    return s.upper()


def apply_map(fn, data, default=None): # Higher-order function that takes a function and data, applies the function to each item in the data
    result = []
    for item in data:
        if isinstance(item, Number):
            ret = fn(item)
        else:
            ret = default
        result.append(ret)
    return result


nums = [3, 7+2j, 5, "hello", 8, 4]
r = apply_map(square, nums, default=0)
print(r)

names = ["alice", "bob", "charlie", 123, None, "dave", 4.5, "emily"]
r = apply_map(to_upper, names, default="")
print(r)

[9, (45+28j), 25, 0, 64, 16]


AttributeError: 'int' object has no attribute 'upper'

In [58]:
# Using "Duck Typing" approach for the same example

def square(x):
    return x * x

def to_upper(s):
    return s.upper()


def apply_map(fn, data, default=None): # Higher-order function that takes a function and data, applies the function to each item in the data
    result = []
    for item in data:
        try:
            ret = fn(item)
        except:
            ret = default
        result.append(ret)
    return result


nums = [3, 7+2j, 5, "hello", 8, 4]
r = apply_map(square, nums, default=0)
print(r)

names = ["alice", "bob", "charlie", 123, None, "dave", 4.5, "emily"]
r = apply_map(to_upper, names, default="")
print(r)

[9, (45+28j), 25, 0, 64, 16]
['ALICE', 'BOB', 'CHARLIE', '', '', 'DAVE', '', 'EMILY']


In [ ]:
#import xml.etree.ElementTree as ET
#import lxml.etree as ET
#tree = ET.parse("example.xml")

import lxml.html as ET
tree = ET.parse("example.html")

print(tree)
data = tree.getroot()
print(data)
data.find(".//*[@name='Singapore']")

<Element data at 0x116828680>


<Element country at 0x106a0d300>

In [8]:
#import json
import ujson as json

with open("data.json", "w") as f:
    json.dump({"name": "Alice", "age": 30, "city": "NYC"}, f)


---

#### Bridge Pattern

A structural pattern that Separates an abstraction from its implementation so both can vary independently. Think of it as a "has-a" relationship that decouples two orthogonal dimensions.

**When to use:** When you want to avoid a class explosion from combining multiple dimensions (e.g., shapes × colors, devices × remote controls).

In [61]:
from abc import ABC, abstractmethod

# --- IMPLEMENTATION hierarchy (the "bridge" side) ---

class Renderer(ABC):
    @abstractmethod
    def render_circle(self, radius: float) -> str: ...

    @abstractmethod
    def render_square(self, side: float) -> str: ...

class VectorRenderer(Renderer):
    def render_circle(self, radius):
        return f"Drawing circle (r={radius}) as vector paths"

    def render_square(self, side):
        return f"Drawing square (s={side}) as vector paths"

class RasterRenderer(Renderer):
    def render_circle(self, radius):
        return f"Drawing circle (r={radius}) as pixel grid"

    def render_square(self, side):
        return f"Drawing square (s={side}) as pixel grid"


# --- ABSTRACTION hierarchy ---

class Shape(ABC):
    def __init__(self, renderer: Renderer):
        self.renderer = renderer          # <-- the bridge

    @abstractmethod
    def draw(self) -> str: ...

class Circle(Shape):
    def __init__(self, renderer, radius):
        super().__init__(renderer)
        self.radius = radius

    def draw(self):
        return self.renderer.render_circle(self.radius)

class Square(Shape):
    def __init__(self, renderer, side):
        super().__init__(renderer)
        self.side = side

    def draw(self):
        return self.renderer.render_square(self.side)


# Usage
vector = VectorRenderer()
raster = RasterRenderer()

shapes = [
    Circle(vector, 5),
    Circle(raster, 5),
    Square(vector, 4),
    Square(raster, 4),
]

for s in shapes:
    print(s.draw())


Drawing circle (r=5) as vector paths
Drawing circle (r=5) as pixel grid
Drawing square (s=4) as vector paths
Drawing square (s=4) as pixel grid


In [64]:
class Color:
    def __init__(self, name):
        self.color_name = name


class Shape:
    def __init__(self, name):
        self.shape_name = name

# -------------------
class Triangle(Shape):
    def __init__(self):
        super().__init__("Triangle")
        print(f"Shape: {self.shape_name}")

class Red(Color):
    def __init__(self):
        super().__init__("Red")
        print(f"Color: {self.color_name}")

class RedTriangle(Red, Triangle):
    def __init__(self):
        Red.__init__(self)
        Triangle.__init__(self)

RedTriangle()
# --------------------

class RedTriangle:
    def __init__(self):
        self.color = Red()
        self.shape = Triangle()
RedTriangle()

Color: Red
Shape: Triangle
Color: Red
Shape: Triangle


##### Bridge Patterns found in the Python standard library

In [ ]:
import logging

# --- ABSTRACTION axis: Loggers (what you log from) ---
app_logger  = logging.getLogger("app")
db_logger   = logging.getLogger("app.db")
api_logger  = logging.getLogger("app.api")

# --- IMPLEMENTATION axis: Handlers (where logs go) ---
console_handler = logging.StreamHandler()
file_handler    = logging.FileHandler("app.log")
memory_handler  = logging.handlers.MemoryHandler(capacity=100)

# Bridge: attach any handler to any logger freely
app_logger.addHandler(console_handler)
app_logger.addHandler(file_handler)     # same logger, two implementations

db_logger.addHandler(file_handler)      # same handler, different logger
api_logger.addHandler(memory_handler)

# Logger never knows or cares what the handler does with the record
app_logger.warning("Low memory")        # goes to console + file
db_logger.error("Query failed")         # goes to file
api_logger.info("Request received")     # goes to memory buffer

In [ ]:
import io

# --- ABSTRACTION axis: IO interface level ---
# RawIOBase       → raw bytes, no buffering
# BufferedIOBase  → buffered bytes
# TextIOBase      → decoded text

# --- IMPLEMENTATION axis: backing store ---
# FileIO      → backed by a real file on disk
# BytesIO     → backed by an in-memory bytes buffer
# SocketIO    → backed by a network socket

# Same TextIOWrapper (abstraction) over different backing stores (implementation)
file_source   = io.TextIOWrapper(io.FileIO("data.txt", "w"))
memory_source = io.TextIOWrapper(io.BytesIO())

# Both expose identical text interface — client code is the same
for stream in [file_source, memory_source]:
    stream.write("hello bridge\n")   # identical call, different backing store

# --- Bridge in action: BufferedWriter over different raw sources ---
raw_file   = io.FileIO("output.bin", "wb")
raw_memory = io.BytesIO()

buffered_file   = io.BufferedWriter(raw_file)    # abstraction + implementation
buffered_memory = io.BufferedWriter(raw_memory)  # same abstraction, different impl

for buf in [buffered_file, buffered_memory]:
    buf.write(b"binary data")
    buf.flush()

In [65]:
import hashlib

# --- ABSTRACTION: uniform hash interface ---
# .update(), .digest(), .hexdigest(), .copy() — always the same

# --- IMPLEMENTATION: pluggable algorithm backends ---
algorithms = ["md5", "sha1", "sha256", "sha512", "blake2b"]

data = b"hello bridge pattern"

for algo in algorithms:
    h = hashlib.new(algo)        # same interface, different implementation
    h.update(data)
    print(f"{algo:10s} → {h.hexdigest()[:20]}...")

# Or using named constructors — same abstraction, swappable backend
h1 = hashlib.md5()
h2 = hashlib.sha256()
h3 = hashlib.blake2b()

for h in [h1, h2, h3]:
    h.update(data)
    print(h.hexdigest()[:20])    # identical calls, different algorithms

md5        → a0fcc5d39ea39fca7a0a...
sha1       → 763d46f08b92417fcba2...
sha256     → 2aa75ef52db923846dab...
sha512     → 92c5604a4e1cce9b86a0...
blake2b    → 4e75bd21eb17a3d57de4...
a0fcc5d39ea39fca7a0a
2aa75ef52db923846dab
4e75bd21eb17a3d57de4


---
#### Composite Pattern

A Pattern that represents an object that is made of other composite objects, providing an abstract interface to operate on the composite objects.

In [66]:
from abc import ABC, abstractmethod
from typing import List

# --- COMPONENT (common interface for both leaf and composite) ---

class FileSystemItem(ABC):
    def __init__(self, name: str):
        self.name = name

    @abstractmethod
    def size(self) -> int: ...

    @abstractmethod
    def display(self, indent: int = 0) -> str: ...


# --- LEAF ---

class File(FileSystemItem):
    def __init__(self, name: str, file_size: int):
        super().__init__(name)
        self._size = file_size

    def size(self):
        return self._size

    def display(self, indent=0):
        return " " * indent + f"📄 {self.name} ({self._size} KB)"


# --- COMPOSITE (holds children, which can be Files or Folders) ---

class Folder(FileSystemItem):
    def __init__(self, name: str):
        super().__init__(name)
        self._children: List[FileSystemItem] = []

    def add(self, item: FileSystemItem):
        self._children.append(item)

    def size(self):
        return sum(child.size() for child in self._children)   # delegates to children

    def display(self, indent=0):
        lines = [" " * indent + f"📁 {self.name}/"]
        for child in self._children:
            lines.append(child.display(indent + 4))
        return "\n".join(lines)


# Usage — build a tree
root = Folder("project")

src = Folder("src")
src.add(File("main.py", 12))
src.add(File("utils.py", 8))

tests = Folder("tests")
tests.add(File("test_main.py", 5))

root.add(src)
root.add(tests)
root.add(File("README.md", 2))

print(root.display())
print(f"\nTotal size: {root.size()} KB")


📁 project/
    📁 src/
        📄 main.py (12 KB)
        📄 utils.py (8 KB)
    📁 tests/
        📄 test_main.py (5 KB)
    📄 README.md (2 KB)

Total size: 27 KB


##### Examples of Composite patterns in Python standard library

In [67]:
import xml.etree.ElementTree as ET

tree = ET.parse("example.xml")
print(tree)

root = tree.getroot()
root

<Element 'data' at 0x10572c590>

In [71]:
root[0].attrib

{'name': 'Liechtenstein'}

In [72]:
for country in root:
    print(country.attrib["name"])
    for child in country:
        print(" ", child.tag, ":", child.text)

Liechtenstein
  rank : 1
  year : 2008
  gdppc : 141100
  neighbor : None
  neighbor : None
Singapore
  rank : 4
  year : 2011
  gdppc : 59900
  neighbor : None
Panama
  rank : 68
  year : 2011
  gdppc : 13600
  neighbor : None
  neighbor : None


In [10]:
from pathlib import Path

p = Path("/usr")
print(p, type(p))
for child in p.iterdir():
    print(child, type(child))

/usr <class 'pathlib.PosixPath'>
/usr/bin <class 'pathlib.PosixPath'>
/usr/standalone <class 'pathlib.PosixPath'>
/usr/libexec <class 'pathlib.PosixPath'>
/usr/sbin <class 'pathlib.PosixPath'>
/usr/local <class 'pathlib.PosixPath'>
/usr/lib <class 'pathlib.PosixPath'>
/usr/X11 <class 'pathlib.PosixPath'>
/usr/X11R6 <class 'pathlib.PosixPath'>
/usr/share <class 'pathlib.PosixPath'>


---

#### Proxy Pattern

A structural pattern that provides a proxy placeholder for another object. The proxy controls access to the original object, allowing us to perform actions before or after the request gets through the original object.

In [75]:
class LightBulb:
    def __init__(self, location):
        self.location = location

    def on(self):
        print(f"{self.location} light bulb is ON")

    def off(self):
        print(f"{self.location} light bulb is OFF")

    def __str__(self):
        return f"<LightBulb: {self.location}>"
    
class Fan:
    def __init__(self, location):
        self.location = location

    def on(self):
        print(f"{self.location} fan is ON")

    def off(self):
        print(f"{self.location} fan is OFF")

    def __str__(self):
        return f"<Fan: {self.location}>"

class Appliance:
    def __init__(self, device):
        self.device = device
        self.state = "off"
    
    def __str__(self):
        return f"<Appliance: {self.device}, state={self.state}>"
    
    def __getattr__(self, attr):
        if attr == self.state:
            print(f"{self.device} is already {self.state}")
            return lambda: None
        else:
            self.state = attr
            print(f"Performing {attr} on {self.device}")
            return getattr(self.device, attr)
        
        
light = Appliance(LightBulb("Living Room"))
light.on()
light.on()
light.off()
light.off()

Performing on on <LightBulb: Living Room>
Living Room light bulb is ON
<LightBulb: Living Room> is already on
Performing off on <LightBulb: Living Room>
Living Room light bulb is OFF
<LightBulb: Living Room> is already off


In [ ]:
# Example of a ProxyDecorator that wraps the original class and adds extra behavior without modifying the original class

class Appliance:
    def __init__(self, device_class):
        self.device = device_class()
        self.state = "off"

    def __call__(self, location):
        self.device.location = location
        return self

    def __str__(self):
        return f"<Appliance: {self.device}, state={self.state}>"
    
    def __getattr__(self, attr):
        if attr == self.state:
            print(f"{self.device} is already {self.state}")
            return lambda: None
        else:
            self.state = attr
            print(f"Performing {attr} on {self.device}")
            return getattr(self.device, attr)

@Appliance
class LightBulb:
    def __init__(self):
        self.location = "Living Room"

    def on(self):
        print(f"{self.location} light bulb is ON")

    def off(self):
        print(f"{self.location} light bulb is OFF")

    def __str__(self):
        return f"<LightBulb: {self.location}>"

@Appliance
class Fan:
    def __init__(self):
        self.location = "Living Room"

    def on(self):
        print(f"{self.location} fan is ON")

    def off(self):
        print(f"{self.location} fan is OFF")

    def __str__(self):
        return f"<Fan: {self.location}>"

        
light = LightBulb("Living Room")
fan = Fan("Living Room")

light.on()
fan.on()

Performing on on <LightBulb: Living Room>
Living Room light bulb is ON
Performing on on <Fan: Living Room>
Living Room fan is ON


#### Example of MockingProxy pattern

In [78]:
class LightBulb:
    def __init__(self, location):
        self.location = location

    def on(self):
        print(f"{self.location} light bulb is ON")

    def off(self):
        print(f"{self.location} light bulb is OFF")

    def __str__(self):
        return f"<LightBulb: {self.location}>"

class MockingProxy:
    def __init__(self, target):
        self.target = target
        self.valid_calls = []
        self.invalid_calls = []

    def __getattr__(self, attr):
        def wrapper(*args, **kwargs):
            if hasattr(self.target, attr):
                self.valid_calls.append((attr, args, kwargs))
            else:
                self.invalid_calls.append((attr, args, kwargs))
            print(f"MockingProxy: called {attr} with args={args} kwargs={kwargs}")
        return wrapper


light = MockingProxy(LightBulb("Living Room"))
light.on()
light.off()
light.blink()

print(light.valid_calls)
print(light.invalid_calls)

MockingProxy: called on with args=() kwargs={}
MockingProxy: called off with args=() kwargs={}
MockingProxy: called blink with args=() kwargs={}
[('on', (), {}), ('off', (), {})]
[('blink', (), {})]


##### Examples of Proxy Pattern in Python standard library


In [86]:
from unittest.mock import MagicMock, patch

class PaymentGateway:
    def charge(self, amount):
        # Real implementation would hit a payment API
        return {"status": "success", "amount": amount}
    
    def refund(self, transaction_id):
        return {"status": "refunded", "id": transaction_id}

# MagicMock is a Proxy — intercepts ALL access to the real object
mock_gateway = MagicMock(spec=PaymentGateway)

# Control what the proxy returns
mock_gateway.charge.return_value = {"status": "failure", "amount": 100}

result = mock_gateway.charge(100)     # proxy intercepts, returns fake response
print(result)                         # {"status": "failure", "amount": 100}

# Proxy records everything — you can verify interactions
mock_gateway.charge.assert_called_once_with(100)
print(mock_gateway.charge.call_count)  # 1

# patch() is a Proxy factory — swaps real object with proxy at runtime
with patch('__main__.PaymentGateway') as MockGateway:
    MockGateway.return_value.charge.return_value = {"status": "success"}
    gw = PaymentGateway()     # actually returns the proxy
    gw.charge(50)

{'status': 'failure', 'amount': 100}
1


In [87]:
import weakref
import gc

class HeavyResource:
    def __init__(self, name):
        self.name = name
    
    def process(self):
        print(f"Processing with {self.name}")
    
    def __del__(self):
        print(f"{self.name} was garbage collected")

resource = HeavyResource("GPU Buffer")

# weakref.proxy() creates a transparent proxy to the object
# that doesn't prevent garbage collection
proxy = weakref.proxy(resource)

# Use the proxy exactly like the real object
proxy.process()                       # Processing with GPU Buffer
print(proxy.name)                     # GPU Buffer
print(type(proxy))                    # weakproxy

# When the real object is deleted, proxy becomes invalid
del resource
gc.collect()

try:
    proxy.process()                   # ReferenceError — real object is gone
except ReferenceError as e:
    print(f"Proxy invalid: {e}")      # Proxy invalid: weakly-referenced object

Processing with GPU Buffer
GPU Buffer
<class 'weakref.ProxyType'>
GPU Buffer was garbage collected
Proxy invalid: weakly-referenced object no longer exists


In [88]:
class TestContext:
    def __enter__(self):
        print("Entering context")
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Exiting context")
        if exc_type:
            print(f"An exception occurred: {exc_val}")
            return True  # Suppress exception
        
with TestContext() as ctx:
    print("Inside context")
    raise ValueError("Something went wrong")

Entering context
Inside context
Exiting context
An exception occurred: Something went wrong


In [89]:
from contextlib import contextmanager
import contextlib

# contextmanager wraps a generator and proxies the context manager protocol
@contextmanager
def managed_resource(name):
    print(f"Acquiring {name}")
    resource = {"name": name, "data": [1, 2, 3]}   # real object
    try:
        yield resource                               # proxy wraps the generator
    finally:
        print(f"Releasing {name}")

# _GeneratorContextManager proxies __enter__ and __exit__
# to the generator's send() and throw() methods
with managed_resource("DB Connection") as res:
    print(res)      # {'name': 'DB Connection', 'data': [1, 2, 3]}


Acquiring DB Connection
{'name': 'DB Connection', 'data': [1, 2, 3]}
Releasing DB Connection


In [ ]:

# contextlib.closing — a Proxy that adds __exit__ to objects that lack it
class LegacyResource:
    def read(self): return "data"
    def close(self): print("Legacy resource closed")
    # No __enter__ / __exit__ — can't use with `with` directly!

# closing() proxies the context manager protocol onto the legacy object
with contextlib.closing(LegacyResource()) as res:
    print(res.read())   # data
# Legacy resource closed  ← proxy called .close() on __exit__

In [96]:
def square(x):
    print(f"Calculating square of {x}...")
    from time import sleep
    sleep(1)  # Simulate expensive computation
    return x * x

values = [2, 3, 3, 2, 3, 3, 3, 4, 5, 2, 2, 2, 3, 3, 3, 4, 5]
result = [square(x) for x in values]
print(result)

Calculating square of 2...
Calculating square of 3...
Calculating square of 3...
Calculating square of 2...
Calculating square of 3...
Calculating square of 3...
Calculating square of 3...
Calculating square of 4...
Calculating square of 5...
Calculating square of 2...
Calculating square of 2...
Calculating square of 2...
Calculating square of 3...
Calculating square of 3...
Calculating square of 3...
Calculating square of 4...
Calculating square of 5...
[4, 9, 9, 4, 9, 9, 9, 16, 25, 4, 4, 4, 9, 9, 9, 16, 25]


In [95]:
from functools import lru_cache

@lru_cache(maxsize=15)  # Cache results of square() for all inputs
def square(x):
    print(f"Computing square({x})")  # Only prints for unique inputs
    from time import sleep
    sleep(1)  # Simulate expensive computation
    return x * x

values = [2, 3, 3, 2, 3, 3, 3, 4, 5, 2, 2, 2, 3, 3, 3, 4, 5]
result = [square(x) for x in values]
print(result)

Computing square(2)
Computing square(3)
Computing square(4)
Computing square(5)
[4, 9, 9, 4, 9, 9, 9, 16, 25, 4, 4, 4, 9, 9, 9, 16, 25]


In [ ]:
# A caching Proxy example: functools.lru_cache and functools.cached_property
# This is also a decorator and a memoize pattern, but it uses a Proxy to transparently cache results.

import functools
import time

# lru_cache wraps a function and proxies calls through a cache
@functools.lru_cache(maxsize=128)
def expensive_computation(n):
    print(f"  [computing {n}]")
    time.sleep(5)                   # simulate expensive work
    return n * n

print(expensive_computation(5))       # [computing 5] → 25
print(expensive_computation(5))       # cache hit → 25  (no recompute)
print(expensive_computation(6))       # [computing 6] → 36

# The proxy exposes cache internals
info = expensive_computation.cache_info()
print(info)   # CacheInfo(hits=1, misses=2, maxsize=128, currsize=2)
expensive_computation.cache_clear()   # proxy control method

# cached_property — same idea, but for object's attributes
class DataProcessor:
    def __init__(self, data):
        self.data = data

    @functools.cached_property         # proxies first access, caches result
    def summary(self):
        print("  [computing summary]")
        return {"count": len(self.data), "sum": sum(self.data)}

dp = DataProcessor([1, 2, 3, 4, 5])
print(dp.summary)   # [computing summary] → {'count': 5, 'sum': 15}
print(dp.summary)   # cached → {'count': 5, 'sum': 15}  (no recompute)

---

#### Facade Pattern

Provide a simplified interface using a scaffolding function / methods of a object that takes care intricacies of initializing
and working with inner composite objects.


In [ ]:
class Lights:

    def on(self):
        print("Turning on the lights.")

    def off(self):
        print("Turning off the lights.")

    def set_brightness(self, level):
        print(f"Setting brightness to {level}.")

class Fan:
    def on(self):
        print("Turning on the fan.")

    def off(self):
        print("Turning off the fan.")
    
    def set_speed(self, speed):
        print(f"Setting fan speed to {speed}.")

class Room:
    def __init__(self):
        self.lights = Lights()
        self.fan = Fan()
   
    def enter(self):
        self.lights.on()
        self.fan.on()

    def exit(self):
        self.lights.off()
        self.fan.off()

    def dim(self):
        self.lights.set_brightness(30)

    def brighten(self):
        self.lights.set_brightness(100)

    def cool(self):
        self.fan.set_speed(5)

    def warm(self):
        self.fan.set_speed(1)

room = Room()
room.enter()
room.dim()
room.cool()
room.brighten()
room.exit()

Turning on the lights.
Turning on the fan.
Setting brightness to 30.
Setting fan speed to 5.
Setting brightness to 100.
Turning off the lights.
Turning off the fan.


In [15]:
from facade_example import HomeTheaterFacade

theater = HomeTheaterFacade()
theater.watch_movie("Inception")
theater.end_movie()


--- Getting ready for the movie! ---
Lights: Dimming for the show...
Projector: Warming up...
Projector: Setting input to Streaming Box
Amp: Powering on...
Amp: Volume set to 15
App: Authenticating user...
App: Now streaming 'Inception'
--- Enjoy your film! ---


--- Shutting down theater... ---
App: Logging out.
Amp: Shutting down...
Projector: Cooling down...
Lights: Turning back up.
Theater is off.


##### Facade pattern examples in Python standard library

In [ ]:
import logging

# Facade — one line, just works
logging.warning("Something went wrong")
logging.info("App started")

# What the Facade is hiding underneath:
logger   = logging.getLogger("root")       # Logger object
handler  = logging.StreamHandler()         # Handler
formatter = logging.Formatter('%(levelname)s:%(name)s:%(message)s')
handler.setFormatter(formatter)            # attach formatter
logger.addHandler(handler)                 # attach handler
logger.setLevel(logging.WARNING)

logger.warning("Something went wrong")     # now call it

In [ ]:
from pathlib import Path

p = Path("/tmp/data")

# Each of these is a Facade call hiding the module it delegates to:
p.mkdir(parents=True, exist_ok=True)    # os.makedirs()
p.exists()                              # os.path.exists()
p.stat().st_size                        # os.stat()
p.read_text()                           # open() + .read() + .close()
p.write_text("hello")                   # open() + .write() + .close()
(p / "sub" / "file.txt").resolve()      # os.path.join() + os.path.abspath()

for f in p.glob("**/*.py"):             # os.walk() + fnmatch
    print(f)

In [14]:
import urllib.request

# Facade — one call
response = urllib.request.urlopen("https://httpbin.org/get")
print(response.read().decode())

# What urlopen() hides:
opener  = urllib.request.OpenerDirector()
opener.add_handler(urllib.request.HTTPHandler())
opener.add_handler(urllib.request.HTTPSHandler())
opener.add_handler(urllib.request.HTTPCookieProcessor())
opener.add_handler(urllib.request.HTTPRedirectHandler())
opener.add_handler(urllib.request.UnknownHandler())
request = urllib.request.Request("https://httpbin.org/get")
response = opener.open(request)         # now actually send
print(response.read().decode())

{
  "args": {}, 
  "headers": {
    "Accept-Encoding": "identity", 
    "Host": "httpbin.org", 
    "User-Agent": "Python-urllib/3.12", 
    "X-Amzn-Trace-Id": "Root=1-6a04b039-36db333641a165871098cc96"
  }, 
  "origin": "49.204.143.183", 
  "url": "https://httpbin.org/get"
}

{
  "args": {}, 
  "headers": {
    "Accept-Encoding": "identity", 
    "Host": "httpbin.org", 
    "User-Agent": "Python-urllib/3.12", 
    "X-Amzn-Trace-Id": "Root=1-6a04b03a-4b8e02f719b0f4b727bc9e05"
  }, 
  "origin": "49.204.143.183", 
  "url": "https://httpbin.org/get"
}



In [18]:
import subprocess

# Facade — clean and simple
result = subprocess.run(
    ["uname", "-a"],
    capture_output=True,
    text=True
)
print(result.stdout)

# What run() hides — the raw Popen machinery:
proc = subprocess.Popen(
    ["uname", "-a"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
stdout, stderr = proc.communicate()     # handle blocking I/O
proc.wait()                             # wait for exit
result = stdout.decode()
print(result)

Darwin Telesto.local 25.4.0 Darwin Kernel Version 25.4.0: Thu Mar 19 19:33:25 PDT 2026; root:xnu-12377.101.15~1/RELEASE_ARM64_T6041 arm64

Darwin Telesto.local 25.4.0 Darwin Kernel Version 25.4.0: Thu Mar 19 19:33:25 PDT 2026; root:xnu-12377.101.15~1/RELEASE_ARM64_T6041 arm64



---

#### Flyweight Pattern

Flywieght Patterns are used to consolidate and underlying optimize memory usage of attributes of multiple instances of a class using a federated (consolidated) backend implementation.

In [ ]:
class String:
    def __init__(self, max_length):
        self.max_length = max_length
        self.values = {}

    def __get__(self, instance, owner=None):
        _id = id(instance)
        if _id not in self.values:
            raise AttributeError("value not set")
        return self.values[_id]
    
    def __set__(self, instance, value):
        _id = id(instance)
        if type(value) is not str:
            raise TypeError("value must be a string")
        if len(value) > self.max_length:
            raise ValueError(f"value exceeds maximum length of {self.max_length}")
        self.values[_id] = value

class Number:
    def __init__(self, range):
        self.range = range
        self.values = {}

    def __get__(self, instance, owner=None):
        _id = id(instance)
        if _id not in self.values:
            raise AttributeError("value not set")
        return self.values[_id]
    
    def __set__(self, instance, value):
        _id = id(instance)
        if type(value) is not int:
            raise TypeError("value must be a number")
        if value not in self.range:
            raise ValueError(f"value {value} out of range {self.range}")
        self.values[_id] = value

class User:
    name = String(max_length=15)
    age = Number(range(0, 150))

    def __init__(self, name=None, age=None):
        if name is not None:
            self.name = name
        if age is not None:
            self.age = age
        
    def __repr__(self):
        return f"User(name='{self.name}', age={self.age})"
    

u1 = User()
u1.name = "Chandra"
u1.age = 30
print(u1)

u2 = User(name="Alice", age=25)
print(u2)

u3 = User(name="Bob", age=30)
print(u3)

print(User.__dict__['name'].values)

User(name='Chandra', age=30)
User(name='Alice', age=25)
User(name='Bob', age=30)
{5705833200: 'Chandra', 5705840592: 'Alice', 5705834064: 'Bob'}


In [ ]:
from user_flyweight import User

u1 = User("Alice")
u1.role = "admin"

u2 = User("Bob")
u2.role = "user"

u3 = User("Charlie")
u3.role = "user"

print(u1, u2, u3, sep="\n")

User({'name': 'Alice', 'role': 'admin'})
User({'name': 'Bob', 'role': 'user'})
User({'name': 'Charlie', 'role': 'user'})


##### Flyweight patterns in the Python standard library

In [ ]:
# Small Integer Caching — The most fundamental Flyweight in Python
# Python pre-creates and reuses integer objects in the range -5 to 256 at interpreter startup.

import sys

# Flyweight — same object reused, not recreated
a = 100
b = 100
print(a is b)           # True — exact same object in memory
print(id(a) == id(b))   # True

# Outside the cache range — new objects created each time
x = 1000
y = 1000
print(x is y)           # False — different objects (CPython)
print(id(x) == id(y))   # False

# The cache saves enormous memory in real programs
# where small integers are used millions of times
nums = [1, 1, 1, 1, 1]
print(all(n is nums[0] for n in nums))   # True — all point to same object!

# Verify with sys.getrefcount
print(sys.getrefcount(1))    # very high — shared by entire runtime
print(sys.getrefcount(1000)) # low — freshly created

In [ ]:
# String Interning — Flyweight for string literals
# Python automatically interns string literals that look like identifiers,
# and you can force interning via sys.intern().

import sys

# Auto-interned — compile-time string literals
a = "hello"
b = "hello"
print(a is b)           # True — same interned object

# Strings that look like identifiers are auto-interned
s1 = "python_rocks"
s2 = "python_rocks"
print(s1 is s2)         # True (CPython)

# Strings with spaces are NOT auto-interned
s3 = "hello world"
s4 = "hello world"
print(s3 is s4)         # False — separate objects

# Force interning with sys.intern() — explicit Flyweight
s5 = sys.intern("hello world")
s6 = sys.intern("hello world")
print(s5 is s6)         # True — now the same object

# Real-world benefit: dictionary key lookups become pointer comparisons
# instead of character-by-character string comparisons
large_dict = {sys.intern(f"key_{i}"): i for i in range(10000)}

# This lookup is O(1) pointer comparison, not O(n) string comparison
key = sys.intern("key_5000")
print(large_dict[key])   # 5000

#### Memoize Pattern

Provides a caching mechanism for function calls (mostly pure functions)

In [31]:
from time import sleep, time

def square(x):
    sleep(1)
    return x * x

values = [2, 2, 5, 2, 6, 3, 6, 2, 2, 3, 3, 5, 6, 6, 2, 3, 5]

start = time()
results = [square(v) for v in values]
end = time()
print(f"Time taken without caching: {end - start} seconds")

Time taken without caching: 17.064401149749756 seconds


In [33]:
from time import sleep, time

def cached(func):
    cache = {}
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    return wrapper

@cached
def square(x):
    sleep(1)
    return x * x

values = [2, 2, 5, 2, 6, 3, 6, 2, 2, 3, 3, 5, 6, 6, 2, 3, 5]

start = time()
results = [square(v) for v in values]
end = time()
print(f"Time taken with caching: {end - start} seconds")

Time taken with caching: 4.009718179702759 seconds


In [38]:
from time import sleep, time

from functools import cache, lru_cache


@lru_cache(maxsize=4)
def square(x):
    print("Square invoked with", x)
    sleep(1)
    return x * x

values = [2, 2, 5, 2, 6, 3, 6, 2, 2, 3, 3, 5, 6, 6, 2, 3, 5, 4, 12, 78, 14, 2]

start = time()
results = [square(v) for v in values]
end = time()
print(f"Time taken with caching: {end - start} seconds")

Square invoked with 2
Square invoked with 5
Square invoked with 6
Square invoked with 3
Square invoked with 4
Square invoked with 12
Square invoked with 78
Square invoked with 14
Square invoked with 2
Time taken with caching: 9.025467872619629 seconds


In [ ]:
import requests

res = requests.get("https://python.org/")
res.status_code
d1 = res.text

In [ ]:
import cached_requests as requests # cached_requests.py should implement caching

res = requests.get("https://python.org/")
res.status_code
d2 = res.text

Fetching URL from cache: https://python.org/


In [ ]:
requests.get("https://chandrashekar.info/")

Fetching URL from cache: https://chandrashekar.info/


<Response [200]>

In [ ]:
requests.requests_cache

{'https://python.org/': <Response [200]>,
 'https://chandrashekar.info/': <Response [200]>}

#### Exercise: Implement a ProxyPath class that also implements a [] operator to access child elements (subfolders and files) within a path, support len() and iterability.


### Descriptor Pattern (Python-specific)

It allows creating rules for attributes using attribute descriptors.
Attribute descriptors are classes with `__get__()` and `__set__()` implemented.

This pattern allows clean separation of logic for applying validation rules for attributes.

In [92]:
class Celsius:  
    store = {}

    def __get__(self, instance, owner=None):
        print("Celsius.__get__ invoked: instance =", instance, ", owner =", owner)
        if instance in self.store:
            return self.store[instance]
        else:
            return 0.0
    
    def __set__(self, instance, value):
        print("Celsius.__set__ invoked: instance =", instance, ", value =", value)
        if value < -273.15:
            raise ValueError("Temperature below -273.15 is not possible.")
        self.store[instance] = value

class Temperature:
    value = Celsius()

temp = Temperature()
print(temp.value)

temp.value = 25
print(temp.value)


Celsius.__get__ invoked: instance = <__main__.Temperature object at 0x1455c4500> , owner = <class '__main__.Temperature'>
0.0
Celsius.__set__ invoked: instance = <__main__.Temperature object at 0x1455c4500> , value = 25
Celsius.__get__ invoked: instance = <__main__.Temperature object at 0x1455c4500> , owner = <class '__main__.Temperature'>
25


In [12]:
# Older approach

class Temperature:
    def __init__(self):
        self._value = 0.0
    
    @property
    def value(self):
        return self._value
    
    @value.setter
    def value(self, new_value):
        if type(new_value) not in (int, float):
            raise TypeError("Temperature value must be a number.")
        if new_value < -273.15:
            raise ValueError("Temperature below -273.15 is not possible.")
        self._value = new_value
    
temp = Temperature()
print(temp.value)
temp.value = 25
print(temp.value)
#temp.value = "hello"

0.0
25


In [102]:
# Newer approach

class Celsius:
    def __init__(self):
        self._values = {}

    def __get__(self, instance, owner=None):
        return self._values.get(instance, 0.0)
    
    def __set__(self, instance, value):
        if type(value) not in (int, float):
            raise TypeError("Temperature value must be a number.")
        if value < -273.15:
            raise ValueError("Temperature below -273.15 is not possible.")
        self._values[instance] = value

class Temperature:
    value = Celsius()

t1 = Temperature()
t2 = Temperature()

t1.value = 10
t2.value = 20
print(t1.value)
print(t2.value)

print(t1.__dict__)
print(t1.__dict__)

10
20
{}
{}


In [106]:
Temperature.__dict__['value']._values

{<__main__.Temperature at 0x1455b53a0>: 10,
 <__main__.Temperature at 0x1455b4950>: 20}